In [1]:
import pandas as pd
import numpy as np
from scipy.special import softmax

In [2]:
from pathlib import Path

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler


data_dir = Path(".")
asd_path = data_dir / "Cluster_result.xlsx"
control_path = data_dir / "Control_data.xlsx"

asd = pd.read_excel(asd_path)
control = pd.read_excel(control_path)


In [3]:
feature_cols = ['T_SRS_AWR', 'T_SRS_COG', 'T_SRS_COMM', 'T_SRS_MOT',
       'T_SRS_RRB', 'CBCL_AP_T', 'CBCL_Externalizing_T', 'ASC_PA', 'ASC_AA',
       'ASC_SA', 'ASC_uncertainty','M_SEQ_hypo','M_SEQ_hyper','M_SEQ_seeking']

In [4]:
asd = asd[['Subject ID', 'Cluster'] + feature_cols]
control = control[['Subject ID', 'Cluster'] + feature_cols]

In [5]:
asd.shape,control.shape

((727, 16), (151, 16))

In [6]:
combined = pd.concat([asd, control])

In [7]:
combined

,Subject ID,Cluster,T_SRS_AWR,T_SRS_COG,T_SRS_COMM,T_SRS_MOT,T_SRS_RRB,CBCL_AP_T,CBCL_Externalizing_T,ASC_PA,ASC_AA,ASC_SA,ASC_uncertainty,M_SEQ_hypo,M_SEQ_hyper,M_SEQ_seeking
0,A0003,Moderate,64.0,70.0,71.0,73.0,68.0,59,59,3,3,8,6,2.500000,3.071429,2.769231
1,A0015,Moderate,76.0,76.0,69.0,60.0,64.0,68,66,5,2,5,6,2.333333,2.357143,2.583333
2,A0173,Moderate,70.0,72.0,74.0,60.0,64.0,82,47,8,2,0,8,1.500000,1.428571,1.076923
3,A0215,Moderate,71.0,77.0,72.0,71.0,60.0,73,74,7,3,4,7,1.666667,2.142857,2.153846
4,A0235,Mild,64.0,68.0,70.0,67.0,62.0,61,56,2,0,7,1,1.166667,1.428571,1.692308
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,C0812,Control,51.0,48.0,44.0,40.0,43.0,50,35,4,1,3,1,1.666667,1.428571,1.769231
147,C0823,Control,57.0,52.0,50.0,44.0,41.0,50,41,0,0,0,0,1.166667,1.357143,2.769231
148,C0762,Control,57.0,52.0,51.0,54.0,45.0,50,42,4,0,3,5,1.500000,1.857143,2.230769
149,C0727,Control,60.0,59.0,51.0,52.0,43.0,50,28,1,0,0,0,1.166667,1.214286,1.076923


In [8]:
combined[['T_SRS_AWR', 'T_SRS_COG', 'T_SRS_COMM', 'T_SRS_MOT',
       'T_SRS_RRB', 'CBCL_AP_T', 'CBCL_Externalizing_T', 'ASC_PA', 'ASC_AA',
       'ASC_SA', 'ASC_uncertainty']] = combined[['T_SRS_AWR', 'T_SRS_COG', 'T_SRS_COMM', 'T_SRS_MOT',
       'T_SRS_RRB', 'CBCL_AP_T', 'CBCL_Externalizing_T', 'ASC_PA', 'ASC_AA',
       'ASC_SA', 'ASC_uncertainty']].round(decimals=0)

In [9]:
imputer = KNNImputer(n_neighbors=2)
combined.loc[:, feature_cols] = imputer.fit_transform(combined[feature_cols])

non_control_rows = combined["Cluster"] != "Control"
control_rows = combined["Cluster"] == "Control"

scaler = StandardScaler()
combined = combined.copy()
combined.loc[non_control_rows, feature_cols] = scaler.fit_transform(
    combined.loc[non_control_rows, feature_cols]
)
combined.loc[control_rows, feature_cols] = scaler.transform(
    combined.loc[control_rows, feature_cols]
)

In [10]:
combined

,Subject ID,Cluster,T_SRS_AWR,T_SRS_COG,T_SRS_COMM,T_SRS_MOT,T_SRS_RRB,CBCL_AP_T,CBCL_Externalizing_T,ASC_PA,ASC_AA,ASC_SA,ASC_uncertainty,M_SEQ_hypo,M_SEQ_hyper,M_SEQ_seeking
0,A0003,Moderate,-0.423920,-0.014220,0.366354,0.913563,0.316872,-0.334992,0.351069,-0.554516,0.548355,0.880311,-0.032740,1.080402,1.551574,0.886216
1,A0015,Moderate,0.898311,0.606058,0.171590,-0.269675,-0.014351,0.541048,1.028834,0.085886,0.050037,-0.016866,-0.032740,0.800283,0.256190,0.614739
2,A0173,Moderate,0.237195,0.192539,0.658499,-0.269675,-0.014351,1.903775,-0.810814,1.046489,0.050037,-1.512161,0.416353,-0.600309,-1.427809,-1.585163
3,A0215,Moderate,0.347381,0.709438,0.463735,0.731527,-0.345575,1.027736,1.803423,0.726288,0.548355,-0.315925,0.191807,-0.320190,-0.132425,-0.012467
4,A0235,Mild,-0.423920,-0.220979,0.268972,0.367454,-0.179963,-0.140316,0.060598,-0.874717,-0.946598,0.581252,-1.155473,-1.160546,-1.427809,-0.686480
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,C0812,Control,-1.856337,-2.288574,-2.262954,-2.090041,-1.753273,-1.211031,-1.972698,-0.234315,-0.448280,-0.614984,-1.155473,-0.320190,-1.427809,-0.574144
147,C0823,Control,-1.195222,-1.875055,-1.678663,-1.725968,-1.918885,-1.211031,-1.391756,-1.515119,-0.946598,-1.512161,-1.380020,-1.160546,-1.557347,0.886216
148,C0762,Control,-1.195222,-1.875055,-1.581281,-0.815785,-1.587661,-1.211031,-1.294932,-0.234315,-0.946598,-0.614984,-0.257287,-0.600309,-0.650579,0.099868
149,C0727,Control,-0.864664,-1.151397,-1.581281,-0.997821,-1.753273,-1.211031,-2.650463,-1.194918,-0.946598,-1.512161,-1.380020,-1.160546,-1.816424,-1.585163


In [ ]:
combined.to_excel(data_dir / "Transformed.xlsx", index=False)

In [ ]:
cluster_order = ["Control", "Mild", "Moderate", "Severe"]
cluster_scores = {label: score for score, label in enumerate(cluster_order)}
missing_clusters = sorted(set(cluster_order) - set(combined["Cluster"].dropna().unique()))
if missing_clusters:
    raise ValueError(f"Missing clusters in combined: {missing_clusters}")

cluster_centers = (
    combined.groupby("Cluster")[feature_cols]
    .mean()
    .reindex(cluster_order)
)

control_center = cluster_centers.loc["Control"].to_numpy(dtype=float)
cluster_distances = {"Control": 0.0}
for label in cluster_order[1:]:
    group_center = cluster_centers.loc[label].to_numpy(dtype=float)
    cluster_distances[label] = float(np.linalg.norm(group_center - control_center))

feature_matrix = combined[feature_cols].to_numpy(dtype=float)
center_matrix = cluster_centers.to_numpy(dtype=float)
diff = feature_matrix[:, None, :] - center_matrix[None, :, :]
squared_distances = np.sum(diff ** 2, axis=2)
score_vector = np.array([cluster_scores[label] for label in cluster_order], dtype=float)

tau_values = [1.0]
soft_score_cols = []
for tau in tau_values:
    probabilities = softmax(-squared_distances / tau, axis=1)
    tau_label = f"{tau:.1f}"
    score_col = f"combined_tau_{tau_label}"
    combined[score_col] = probabilities @ score_vector
    soft_score_cols.append(score_col)

distance_summary = pd.DataFrame(
    [
        {label: cluster_distances[label] for label in cluster_order}
    ]
)

soft_index_path = data_dir / "ASD_index.xlsx"
with pd.ExcelWriter(soft_index_path, engine="openpyxl") as writer:
    combined.to_excel(writer, sheet_name="soft_index", index=False)
    cluster_centers.reset_index().to_excel(
        writer,
        sheet_name="cluster_feature_means",
        index=False,
    )
    distance_summary.to_excel(
        writer,
        sheet_name="distance_to_control",
        index=False,
    )

combined[["Subject ID", "Cluster", *soft_score_cols]].head()

,Subject ID,Cluster,combined_tau_1.0
0,A0003,Moderate,2.042533
1,A0015,Moderate,2.001416
2,A0173,Moderate,1.996455
3,A0215,Moderate,2.005170
4,A0235,Mild,1.296269


In [13]:
cluster_centers

,T_SRS_AWR,T_SRS_COG,T_SRS_COMM,T_SRS_MOT,T_SRS_RRB,CBCL_AP_T,CBCL_Externalizing_T,ASC_PA,ASC_AA,ASC_SA,ASC_uncertainty,M_SEQ_hypo,M_SEQ_hyper,M_SEQ_seeking
Cluster,,,,,,,,,,,,,,
Control,-1.151804,-1.516649,-1.623523,-1.101799,-1.369953,-1.006041,-0.993561,-0.469695,-0.642987,-0.680341,-0.795604,-0.800658,-0.914802,-0.687967
Mild,-0.708841,-0.867488,-0.911237,-0.723408,-0.939737,-0.826402,-0.746506,-0.356183,-0.541250,-0.456527,-0.626783,-0.551184,-0.669623,-0.649132
Moderate,0.181923,0.236653,0.276268,0.137981,0.211679,0.186891,0.234376,-0.037188,-0.128513,-0.057779,0.011877,0.072158,0.053401,0.087238
Severe,0.882363,1.051543,1.048667,0.996797,1.229370,1.079607,0.842831,0.703116,1.213871,0.921629,1.081129,0.826084,1.072793,0.968324
